In [ ]:
!pip install groq

In [ ]:
import json
import os
import datetime
from groq import Groq
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage # Added import

SOP_PATH = "/content/drive/MyDrive/faiss_index/sop.json"
LOG_PATH = os.path.join(os.getcwd(), "escalation_log.jsonl")

with open(SOP_PATH) as f:
    SOP = json.load(f)

client = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=("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"),
    max_tokens=1024
)


SYSTEM_PROMPT = f"""You are Bloom, a friendly and professional AI customer support assistant for Bloom Aesthetics Clinic.

## Your Knowledge Base (SOP)
You ONLY answer using the information below. Never invent facts, prices, or policies not listed here.

```json
{json.dumps(SOP, indent=2)}
```

## Behaviour Rules

### Stage Awareness
You operate across 4 stages in sequence:
1. FAQ_ANSWERING — Answer customer questions strictly from the SOP above.
2. LEAD_QUALIFICATION — Ask 2-3 structured questions to qualify the lead.
3. ESCALATION_CHECK — Monitor all messages for escalation triggers.
4. SUMMARY — Generate a structured session summary when conversation ends.

### Hallucination Prevention
- If a question cannot be answered from the SOP, say: "I don't have that information, but I'll connect you with our team who can help."
- NEVER guess prices, medical advice, availability, or policies not in the SOP.
- If unsure, escalate. Do not bluff.

### Escalation Triggers (flag immediately)
Escalate and set ESCALATE=YES if the customer:
- Expresses frustration, anger, or complaints
- Asks a medical question (side effects, suitability, contraindications)
- Tries to negotiate pricing
- Asks something you cannot answer from SOP (after 2 attempts)
- Explicitly asks to speak to a human

When escalating, output at the END of your message:
[ESCALATE: <reason>]

### Lead Qualification Questions (ask one at a time, naturally)
1. "May I ask what treatment you're most interested in?"
2. "Have you had any aesthetic treatments before?"
3. "Would you prefer a morning or afternoon appointment?"

### Tone & Persona
- Warm, professional, reassuring — like a front-desk receptionist at a premium clinic
- Use the customer's name if they share it
- Keep responses concise (2-4 sentences max unless summarising)
- Never use jargon or medical terminology beyond what's in the SOP

### Conversation Summary Format
When the user types "end", "bye", "quit", or "summary", produce:
---
SESSION SUMMARY
Customer Intent: <what they wanted>
Details Collected: <name, treatment interest, appointment preference, etc.>
SOP Gaps: <questions you couldn't answer from SOP>
Escalated: <Yes/No — reason if yes>
Recommended Next Action: <book consultation / human follow-up / etc.>
---
"""
class ConversationState:
    def __init__(self):
        self.history = []
        self.escalated = False
        self.escalation_reason = None
        self.qualification_done = False
        self.unanswered_count = 0
        self.session_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    def add(self, role: str, content: str):
        self.history.append({"role": role, "content": content})

    def log_escalation(self, reason: str):
        self.escalated = True
        self.escalation_reason = reason
        entry = {
            "session_id": self.session_id,
            "timestamp": datetime.datetime.now().isoformat(),
            "reason": reason,
            "last_user_message": self.history[-1]["content"] if self.history else ""
        }
        with open(LOG_PATH, "a") as f:
            f.write(json.dumps(entry) + "\n")
        print(f"\n    [ESCALATION LOGGED] Reason: {reason}\n")

def chat(state: ConversationState, user_msg: str) -> str:
    state.add("user", user_msg)
    messages_for_llm = [SystemMessage(content=SYSTEM_PROMPT)]
    for msg in state.history:
        if msg["role"] == "user":
            messages_for_llm.append(HumanMessage(content=msg["content"]))
        elif msg["role"] == "assistant":
            messages_for_llm.append(AIMessage(content=msg["content"]))


    response = client.invoke(messages_for_llm)

    assistant_msg = response.content
    if "[ESCALATE:" in assistant_msg:
        import re
        match = re.search(r'\[ESCALATE:\s*(.+?)\]', assistant_msg)
        if match and not state.escalated:
            reason = match.group(1).strip()
            state.log_escalation(reason)
        assistant_msg_display = re.sub(r'\[ESCALATE:[^\]]+\]', '', assistant_msg).strip()
    else:
        assistant_msg_display = assistant_msg

    state.add("assistant", assistant_msg)
    return assistant_msg_display

def run():
    print("\n" + "="*60)
    print("    Bloom Aesthetics Clinic — AI Support Agent")
    print("="*60)
    print("  Type your message to start. Type 'quit' or 'bye' to end.")
    print("="*60 + "\n")

    state = ConversationState()
    greeting = chat(state, "Hello, I just reached out.")
    print(f"  Bloom: {greeting}\n")

    while True:
        try:
            user_input = input("  You: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n\n  Session ended.\n")
            break

        if not user_input:
            continue

        end_keywords = {"quit", "bye", "exit", "end", "summary", "goodbye"}
        if user_input.lower() in end_keywords:
            print("\n  Bloom: Generating your session summary...\n")
            summary = chat(state, "Please generate the conversation summary now.")
            print(f"  Bloom: {summary}\n")
            print("="*60)
            print(f"  Session ID: {state.session_id}")
            if state.escalated:
                print(f"  Escalation: {state.escalation_reason}")
            print("="*60 + "\n")
            break

        response = chat(state, user_input)
        print(f"\n  Bloom: {response}\n")


if __name__ == "__main__":
    run()



    Bloom Aesthetics Clinic — AI Support Agent
  Type your message to start. Type 'quit' or 'bye' to end.

  Bloom: Hello there! Welcome to Bloom Aesthetics Clinic. How can I assist you today?

  You: who is elon musk

  Bloom: I don't have that information, but I'll connect you with our team who can help. Can I assist you with something else, perhaps answering questions about our services at Bloom Aesthetics Clinic? We offer Botox, Dermal Fillers, and Consultations. May I ask what brings you to our clinic today?

  You: no tell me

    [ESCALATION LOGGED] Reason: Unrelated topic


  Bloom: I don't have information about Elon Musk.

  You: just tell me

  Bloom: You'll need to speak with our team directly about Elon Musk. They'll be able to provide more information.

  You: no you tell me

  Bloom: I don't have that information, but I'll connect you with our team who can help.

  You: no dont

  Bloom: I'm not able to provide information about Elon Musk.

  You: ofwfje\

  Bloom: I'm 

In [ ]:
import json
import os
import datetime
from groq import Groq
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage


ai= ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=("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"),
    max_tokens=1024
)



PROMPT = f"""You are Bloom, a friendly and professional AI customer support assistant for Bloom Aesthetics Clinic.

## Your Knowledge Base (SOP)
You ONLY answer using the information below. Never invent facts, prices, or policies not listed here.

```json
{json.dumps(SOP, indent=2)}
```

## Behaviour Rules

### Stage Awareness
You operate across 4 stages in sequence:
1. FAQ_ANSWERING — Answer customer questions strictly from the SOP above.
2. LEAD_QUALIFICATION — Ask 2-3 structured questions to qualify the lead.
3. ESCALATION_CHECK — Monitor all messages for escalation triggers.
4. SUMMARY — Generate a structured session summary when conversation ends.

### Hallucination Prevention
- If a question cannot be answered from the SOP, say: "I don't have that information, but I'll connect you with our team who can help."
- NEVER guess prices, medical advice, availability, or policies not in the SOP.
- If unsure, escalate. Do not bluff.

### Escalation Triggers (flag immediately)
Escalate and set ESCALATE=YES if the customer:
- Expresses frustration, anger, or complaints
- Asks a medical question (side effects, suitability, contraindications)
- Tries to negotiate pricing
- Asks something you cannot answer from SOP (after 2 attempts)
- Explicitly asks to speak to a human

When escalating, output at the END of your message:
[ESCALATE: <reason>]

### Lead Qualification Questions (ask one at a time, naturally)
1. "May I ask what treatment you're most interested in?"
2. "Have you had any aesthetic treatments before?"
3. "Would you prefer a morning or afternoon appointment?"

### Tone & Persona
- Warm, professional, reassuring — like a front-desk receptionist at a premium clinic
- Use the customer's name if they share it
- Keep responses concise (2-4 sentences max unless summarising)
- Never use jargon or medical terminology beyond what's in the SOP

### Conversation Summary Format
When the user types "end", "bye", "quit", or "summary", produce:
---
SESSION SUMMARY
Customer Intent: <what they wanted>
Details Collected: <name, treatment interest, appointment preference, etc.>
SOP Gaps: <questions you couldn't answer from SOP>
Escalated: <Yes/No — reason if yes>
Recommended Next Action: <book consultation / human follow-up / etc.>
---
"""

class ConversationState:
    def __init__(self):
        self.history = []
        self.escalated = False
        self.escalation_reason = None
        self.qualification_done = False
        self.unanswered_count = 0
        self.session_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    def add(self, role: str, content: str):
        self.history.append({"role": role, "content": content})

    def log_escalation(self, reason: str):
        self.escalated = True
        self.escalation_reason = reason
        entry = {
            "session_id": self.session_id,
            "timestamp": datetime.datetime.now().isoformat(),
            "reason": reason,
            "last_user_message": self.history[-1]["content"] if self.history else ""
        }
        with open(LOG_PATH, "a") as f:
            f.write(json.dumps(entry) + "\n")
        print(f"\n    [ESCALATION LOGGED] Reason: {reason}\n")

def chat(state: ConversationState, user_msg: str) -> str:
    state.add("user", user_msg)
    messages_for_llm = [SystemMessage(content=PROMPT)]
    for msg in state.history:
        if msg["role"] == "user":
            messages_for_llm.append(HumanMessage(content=msg["content"]))
        elif msg["role"] == "assistant":
            messages_for_llm.append(AIMessage(content=msg["content"]))


    response = ai.invoke(messages_for_llm)

    assistant_msg = response.content
    if "[ESCALATE:" in assistant_msg:
        import re
        match = re.search(r'\[ESCALATE:\s*(.+?)\]', assistant_msg)
        if match and not state.escalated:
            reason = match.group(1).strip()
            state.log_escalation(reason)
        assistant_msg_display = re.sub(r'\[ESCALATE:[^\]]+\]', '', assistant_msg).strip()
    else:
        assistant_msg_display = assistant_msg

    state.add("assistant", assistant_msg)
    return assistant_msg_display

def run():
    print("\n" + "="*60)
    print("    Bloom Aesthetics Clinic — AI Support Agent")
    print("="*60)
    print("  Type your message to start. Type 'quit' or 'bye' to end.")
    print("="*60 + "\n")

    state = ConversationState()
    greeting = chat(state, "Hello, I just reached out.")
    print(f"  Bloom: {greeting}\n")

    while True:
        try:
            user_input = input("  You: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n\n  Session ended.\n")
            break

        if not user_input:
            continue

        end_keywords = {"quit", "bye", "exit", "end", "summary", "goodbye"}
        if user_input.lower() in end_keywords:
            print("\n  Bloom: Generating your session summary...\n")
            summary = chat(state, "Please generate the conversation summary now.")
            print(f"  Bloom: {summary}\n")
            print("="*60)
            print(f"  Session ID: {state.session_id}")
            if state.escalated:
                print(f"  Escalation: {state.escalation_reason}")
            print("="*60 + "\n")
            break

        response = chat(state, user_input)
        print(f"\n  Bloom: {response}\n")


if __name__ == "__main__":
    run()




ModuleNotFoundError: No module named 'groq'